# Resolve a tea supplier in five minutes

**Start with a useful answer. Escalate only when the evidence is weak.**

This walkthrough uses one fictional but fully local shipment document. It deliberately does **not** claim anything about Unilever or a real supply chain. Replace the document and candidate export with records you control. Arche returns proposed fields and candidates; when no candidate is safe to link, it opens a case and explains the next evidence worth collecting.


## 1. First value: resolve one document against a supplier master

This is the normal happy path. Arche reads local text directly; PDFs, scans, and images use the optional Docling/OCR path. The five lines below extract labelled supplier, distributor, estate, registration, and country fields, then compare the supplier against a caller-owned candidate record.


In [ ]:
from pathlib import Path

from arche import resolve_documents

registry = [{"entity_id": "ent_kijani", "name": "Kijani Tea Exporters Limited", "country": "Kenya"}]
shipment = Path("tea_supplier_shipment.txt")
if not shipment.is_file():
    shipment = Path("examples/notebooks/tea_supplier_shipment.txt")
report = resolve_documents(
    shipment, candidates=registry,
    entity="organisation", extraction_backend="regex", progress=False,
)
print(report.review(reveal=True))


The result is not a silent merge. The candidate decision is a **proposal**; `review()` identifies the extracted field spans and masks values unless you explicitly request them. A reviewer or application must turn reviewed fields into vNext Evidence before a receipt, policy action, or entity-memory change is possible.


## 2. When there is no safe candidate

Different names should not be forced together just because they both handle tea. Here the same document is compared with a different candidate. Arche opens an unresolved `ResolutionCase`, rather than creating a guessed link or treating absence as proof that the organisations are different.


In [ ]:
unresolved = resolve_documents(
    shipment,
    candidates=[{"entity_id": "ent_kericho", "name": "Kericho Highlands Processing", "country": "Kenya"}],
    entity="organisation", extraction_backend="regex", progress=False,
)
print(unresolved.review(reveal=True)["cases"][0])


## 3. Let the case planner say what would change the answer

The front door has already identified the document observation, the candidate, the missing independent registration evidence, and permitted actions. Persist that value-free state in a caller-owned runtime only when you want to plan or execute follow-up work.


In [ ]:
import arche
from arche.runtime import ResolutionBudget, ToolCapability

engine = arche.attach("duckdb:///:memory:")
saved = unresolved.persist(engine)
case_id = saved["case_ids"][0]
plan = engine.plan_case(
    case_id,
    capabilities=(
        ToolCapability("external_registry", ("registry_lookup",), "document-resolution-v1"),
        ToolCapability("caller_document", ("document_extract",), "document-resolution-v1"),
    ),
    budget=ResolutionBudget(1, 1.0),
)
print(plan.actions[0].rationale)


That rationale is the planner's inspectable reasoning: **which uncertainty exists, which action is permitted, and why it is worth its cost.** There is no hidden automatic lookup and no invented relationship.


## 4. Real PDF, scan, or image input

For a document you own, the matching CLI front door emits the same masked review artifact. Add `--store` only when you want to persist its value-free case, Observation, and permitted actions in a local runtime. A persisted `registry_lookup` action can use a caller-owned, policy-pinned HTTPS connector; its query remains transient and its response becomes an Observation.


In [ ]:
# arche resolve-documents tea-shipment.pdf --entity organisation --candidates suppliers.json --store tea.duckdb --out tea-review.json
#
# If the review artifact opens a case and you want to acquire registry evidence:
# arche case registry-lookup CASE_ID REGISTRY_ACTION_ID --connector registry.json --store tea.duckdb
#
# Or, to extract a caller-owned PDF/image through a separately planned local action:
# arche case open tea-shipment.pdf --store tea.duckdb
# arche case plan CASE_ID --enable-local-document
# arche case ingest CASE_ID ACTION_ID tea-shipment.pdf --approved-by analyst-1
# arche case evidence CASE_ID ACTION_ID reviewed-fields.json --review-id review-1
# arche case review CASE_ID --out tea-review.json


`reviewed-fields.json` is caller-owned and can contain fields such as `supplier_name`, `distributor_name`, `registration_id`, `span`, and `page`. Reviewers see the original document in their application; Arche stores the durable evidence references needed to later explain or revise the decision.


## 5. What agents may do next

An optional agent receives the same case assessment and can recommend only already-permitted actions or already-qualified resolver methods. Good recommendations are concrete: “obtain the Kenyan registration identifier,” “compare the distributor's shipment reference,” or “send these two fields to review.” It must not turn an absent match into a positive link.

Splink, RecordLinkage, and domain matchers belong after this basic experience. Arche should select them only when their exact configuration has a completed compatible benchmark qualification. A notebook must never manufacture a qualification from a review pack or a placeholder hash.


## 6. Roadmap from here

1. **Review pane:** render `report.review()` / `arche case review` as a local UI with source document, page/span highlight, candidate comparison, planner recommendation, action outcomes, and explicit reviewer choices.
2. **Tea semantic mapping:** turn reviewed supplier/distributor/estate fields into explicit proposals for claims and relationships, then promote only with independent support and contradiction checks.
3. **Resolver selection:** qualify deterministic, Splink, and domain configurations on complete mappings, then let the planner select only eligible methods under a budget.
4. **Entity memory:** promote accepted, independently supported supplier/distributor/estate claims and relationships; preserve contradictions and open questions for the next shipment.
